## Checking GPU Availability

In [1]:
import faiss
import torch

In [2]:
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("FAISS Version:", faiss.__version__)
print("FAISS GPUs:", faiss.get_num_gpus())

Torch: 2.12.0
CUDA: 12.9
CUDA Available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
FAISS Version: 1.14.1
FAISS GPUs: 1


In [3]:
import os

In [ ]:
os.environ['GOOGLE_API_KEY'] = ''

## Code using LangChain Classic

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationSummaryBufferMemory

D:\Jobs\Internship\PureLogics_CMIT\RAG_Agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
llm = ChatGoogleGenerativeAI(model = 'gemini-3.6-flash')
memory = ConversationSummaryBufferMemory(llm = llm, max_token_limit = 100, return_messages = True)

C:\Users\Darab\AppData\Local\Temp\ipykernel_16780\1680842140.py:2: LangChainDeprecationWarning: The class `ConversationSummaryBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryBufferMemory(llm = llm, max_token_limit = 100, return_messages = True)


In [7]:
conversation = ConversationChain(llm = llm, memory = memory, verbose = True)

C:\Users\Darab\AppData\Local\Temp\ipykernel_16780\2000372045.py:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(llm = llm, memory = memory, verbose = True)


In [8]:
response_1 = conversation.predict(input = 'Hello, My name is ExoAis, this is my Deep Learning internship')
print(f'AI: {response_1}\n')



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: Hello, My name is ExoAis, this is my Deep Learning internship
AI:

> Finished chain.
AI: Hello ExoAis! It is wonderful to meet you, and a huge congratulations on starting your Deep Learning internship! 

Deep Learning is such a fascinating and rapidly evolving field to be working in right now. There are so many interesting areas you might be diving into—are you going to be focusing on Computer Vision with architectures like Convolutional Neural Networks (CNNs) and Vision Transformers (ViTs), or perhaps Natural Language Processing (NLP) with Large Language Models and Attention mechanisms? Or maybe you'll be working on Audio Processing, Reinforcement Learning, 

## Code using Latest LangChain

In [15]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated
from typing_extensions import TypedDict

In [28]:
llm = ChatGoogleGenerativeAI(model = 'gemini-3-flash-preview')

In [29]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def call_model(state: State):
    response = llm.invoke(state['messages'])
    return {'messages': [response]}

workflow = StateGraph(State)
workflow.add_node('chatbot', call_model)
workflow.add_edge(START, 'chatbot')
workflow.add_edge('chatbot', END)

memory = MemorySaver()
conversation = workflow.compile(checkpointer = memory)

In [30]:
config = {'configurable': {'thread_id': 'session_langchain'}}
user_input = {'messages': [('user', 'Hello, My name is ExoAis, this is my Deep Learning internship.')]}
response = conversation.invoke(user_input, config = config)
print(response['messages'][-1].text)

Hello **ExoAis**! It is a pleasure to meet you.

Congratulations on starting your **Deep Learning internship**! That is an incredibly exciting field to be in right now, especially with how fast technology is evolving.

Since I am an AI, you can think of me as a resource or a "virtual mentor" throughout your internship. I can help you with several things:

1.  **Explaining Concepts:** If you're struggling with backpropagation, transformers, vanishing gradients, or specific architectures (CNNs, RNNs, GANs, etc.), I can break them down for you.
2.  **Coding & Debugging:** I can help you write or debug code in **PyTorch, TensorFlow, or Keras**. Just paste your error message or the logic you're trying to implement.
3.  **Research Papers:** If you're assigned to read a complex paper (like *Attention Is All You Need* or *ResNet*), I can help summarize the key points or explain the math behind them.
4.  **Math Foundations:** If the linear algebra or calculus gets a bit heavy, we can walk throu

In [31]:
user_input = {'messages': [('user', 'What is my name?')]}
response = conversation.invoke(user_input, config = config)
print(response['messages'][-1].text)

Your name is **ExoAis**!


## Modern Langchain Agent

In [32]:
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain.agents import create_agent

C:\Users\Darab\AppData\Local\Temp\ipykernel_16780\1329366332.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.agent_toolkits.load_tools import load_tools


In [33]:
llm = ChatGoogleGenerativeAI(model = 'gemini-3-flash-preview')

In [34]:
tools = load_tools(['wikipedia', 'llm-math'], llm = llm)
agent = create_agent(tools = tools, model = llm)

In [35]:
response = agent.invoke({'messages': ['Who is Alan Turing? Write briefly in 3 lines', 'What is 68 * 75?']})

In [57]:
response['messages'][2].content[0]['text']

'Alan Turing was a pioneering British mathematician and computer scientist often considered the father of modern computer science and artificial intelligence.\nDuring WWII, he played a crucial role in cracking the German Enigma code, which significantly shortened the war and saved millions of lives.\nHe also formalized the concepts of algorithms and computation through the Turing machine and proposed the "Turing Test" to measure machine intelligence.\n\n**68 * 75 = 5,100**'

## Vector Databases

In [75]:
import faiss
import pinecone
import re
import PyPDF2
import pdfreader
from PyPDF2 import PdfReader
import pandas as pd
from langchain_classic.text_splitter import CharacterTextSplitter
from langchain_classic.vectorstores import FAISS, VectorStore
from langchain_classic.embeddings import OpenAIEmbeddings
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [59]:
llm = ChatGoogleGenerativeAI(model = 'gemini-3-flash-preview')

In [60]:
doc = PdfReader('CreditCard.pdf')

In [61]:
raw_text = ''
for i, page in enumerate(doc.pages):
    text = page.extract_text()
    if text:
        raw_text += text
print(len(raw_text))

111327


In [62]:
text_splitter = CharacterTextSplitter(separator = '\n\n', chunk_size = 1000, chunk_overlap = 200, length_function = len)

In [63]:
text = text_splitter.split_text(raw_text)

In [64]:
text

["Issued on: 2 December 2016                                                                                             BNM/RH/ PD 034-1   \n   \n                                                                                                                   \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nCredit Card -i \n \n \n \n \n \n \n \n \n \n \n \n \nApplicable to:  \n1. Credit card-i issuers  \n2. Credit card -i acquirers  \n \nIssued on: 2 December 2016                                                                                           \n Table of contents  \n \nPART A  OVERVIEW  ................................ ................................ ...............................  1 \n1 Introduction  ................................ ................................ ................................  1 \n2 Applicability  ................................ ................................ ...............................  1 \n3 Legal provisions  .....................

In [65]:
def clean(s, sep_token = '\n'):
    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'. ,', ' ', s)
    s = s.replace('. .', ',')
    s = s.replace('..', '.')
    s = s.replace('\n', ' ')
    s = s.strip()
    return s
texts = list(map(clean, text))

In [66]:
query1 = 'What is billing cycle?'
query2 = 'What is deal branding?'
embeddings = GoogleGenerativeAIEmbeddings(model = 'models/gemini-embedding-2')
embedding1 = embeddings.embed_query(query1)
embedding2 = embeddings.embed_query(query2)
len(embedding1), len(embedding2)

(3072, 3072)

In [68]:
db_faiss = FAISS.from_texts(texts, embeddings)

In [70]:
faiss_q1 = db_faiss.similarity_search(query1)

In [71]:
faiss_q1[0].page_content

"Issued on: 2 December 2016 BNM/RH/ PD 034-1 Credit Card -i Applicable to: 1. Credit card-i issuers 2. Credit card -i acquirers Issued on: 2 December 2016 Table of contents PART A OVERVIEW ................,...............,............... 1 1 Introduction ................,...............,................ 1 2 Applicability ................,...............,............... 1 3 Legal provisions ................,...............,............ 1 4 Effective date ................,...............,.............. 2 5 Interpretation ................,...............,.............. 2 6 Policy documents superseded ................,...............,. 3 PART B BUSINESS CONDUCT ................,...............,...... 4 7 Shariah concepts ................,...............,........... 4 8 Minimum age and income requirements ................,.......... 6 9 Deposit placement ................,...............,.......... 6 10 Minimum monthly repayment ................,...............,.. 7 11 Guiding principle on f

In [73]:
similar_query1 = db_faiss.similarity_search_with_score(query1)
similar_query1

[(Document(id='f8d047e3-470b-407f-85e4-89e474c5f626', metadata={}, page_content="Issued on: 2 December 2016 BNM/RH/ PD 034-1 Credit Card -i Applicable to: 1. Credit card-i issuers 2. Credit card -i acquirers Issued on: 2 December 2016 Table of contents PART A OVERVIEW ................,...............,............... 1 1 Introduction ................,...............,................ 1 2 Applicability ................,...............,............... 1 3 Legal provisions ................,...............,............ 1 4 Effective date ................,...............,.............. 2 5 Interpretation ................,...............,.............. 2 6 Policy documents superseded ................,...............,. 3 PART B BUSINESS CONDUCT ................,...............,...... 4 7 Shariah concepts ................,...............,........... 4 8 Minimum age and income requirements ................,.......... 6 9 Deposit placement ................,...............,.......... 6 10 Minimum m

In [77]:
chain = load_qa_chain(llm, chain_type = 'stuff')

In [78]:
chain.run(input_documents = faiss_q1, question = query1)

C:\Users\Darab\AppData\Local\Temp\ipykernel_16780\2853916181.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain.run(input_documents = faiss_q1, question = query1)


'I don\'t know. The provided document does not contain a definition for the term "billing cycle." It mentions a "12-month cycle" in the context of risk categories (Section 12.2) and "monthly statements" (Section 17.1), but it does not explicitly define what a billing cycle is.'